In [1]:
import pandas as pd
import numpy as np


In [2]:
# FASE 1: EXTRACT (Extracción de Datos)
# ==========================================

print("--- FASE 1: EXTRACT ---")
# Cargamos los datasets desde la carpeta Data (subimos un nivel con ../)
df_listings = pd.read_csv('../Data/listings.csv')
df_calendar = pd.read_csv('../Data/calendar.csv')
df_reviews = pd.read_csv('../Data/reviews.csv')

print(f"Dimensiones iniciales de listings: {df_listings.shape}")
print(f"Dimensiones iniciales de calendar: {df_calendar.shape}")
print(f"Dimensiones iniciales de reviews: {df_reviews.shape}\n")

--- FASE 1: EXTRACT ---


C:\Users\aguub\AppData\Local\Temp\ipykernel_5864\3016533800.py:6: DtypeWarning: Columns (61,62,95) have mixed types. Specify dtype option on import or set low_memory=False.
  df_listings = pd.read_csv('../Data/listings.csv')


Dimensiones iniciales de listings: (23729, 106)
Dimensiones iniciales de calendar: (8661286, 7)
Dimensiones iniciales de reviews: (387099, 6)



In [3]:
# FASE 2: TRANSFORM - Paso T1 (Duplicados y Poda Estructural)
# ===========================================================
print("--- FASE 2: TRANSFORM (Paso T1) ---")

# 1. Eliminación de registros duplicados
filas_antes = df_listings.shape[0]
df_listings = df_listings.drop_duplicates()
filas_despues = df_listings.shape[0]
print(f"Se eliminaron {filas_antes - filas_despues} filas duplicadas en listings.")

# 2. Poda de columnas con nulos masivos (> 50%)
# Calculamos el porcentaje exacto de valores nulos por columna
porcentaje_nulos = (df_listings.isnull().sum() / len(df_listings)) * 100

# Identificamos las columnas que superan el umbral del 50%
columnas_a_eliminar = porcentaje_nulos[porcentaje_nulos > 50].index.tolist()

print(f"\nSe detectaron {len(columnas_a_eliminar)} columnas con más del 50% de nulos:")
print(columnas_a_eliminar)

# Eliminamos estas columnas irrecuperables del dataframe principal
 # df_listings = df_listings.drop(columns=columnas_a_eliminar)

print(f"\nDimensiones de listings después del Paso T1: {df_listings.shape}")

--- FASE 2: TRANSFORM (Paso T1) ---
Se eliminaron 0 filas duplicadas en listings.

Se detectaron 12 columnas con más del 50% de nulos:
['notes', 'access', 'house_rules', 'thumbnail_url', 'medium_url', 'xl_picture_url', 'neighbourhood_group_cleansed', 'square_feet', 'weekly_price', 'monthly_price', 'license', 'jurisdiction_names']

Dimensiones de listings después del Paso T1: (23729, 106)


In [4]:
df_listings = df_listings.drop(columns=columnas_a_eliminar)
print(f"Columnas eliminadas. El dataset ahora tiene {df_listings.shape[1]} columnas.\n")

Columnas eliminadas. El dataset ahora tiene 94 columnas.



In [5]:
# FASE 2: TRANSFORM - Paso T2 (Formateo de Tipos de Datos)
# ==========================================
print("--- FASE 2: TRANSFORM (Paso T2) ---")

# Limpieza de columnas monetarias (quitar '$' y ',' y pasar a float)
columnas_dinero = ['price', 'security_deposit', 'cleaning_fee', 'extra_people']

for col in columnas_dinero:
    # Verificamos si la columna existe y es de tipo string/object
    if col in df_listings.columns and df_listings[col].dtype == 'O':
        df_listings[col] = df_listings[col].str.replace('$', '', regex=False)
        df_listings[col] = df_listings[col].str.replace(',', '', regex=False)
        df_listings[col] = df_listings[col].astype(float)
        print(f"Columna '{col}' convertida a formato numérico (float).")

# Chequeo rápido de cómo quedaron esas columnas
display(df_listings[['price', 'security_deposit', 'cleaning_fee', 'extra_people']].head())

--- FASE 2: TRANSFORM (Paso T2) ---
Columna 'price' convertida a formato numérico (float).
Columna 'security_deposit' convertida a formato numérico (float).
Columna 'cleaning_fee' convertida a formato numérico (float).
Columna 'extra_people' convertida a formato numérico (float).


,price,security_deposit,cleaning_fee,extra_people
0,3983.0,0.0,3319.0,0.0
1,1593.0,NaN,NaN,0.0
2,2987.0,NaN,NaN,0.0
3,2987.0,19914.0,1328.0,0.0
4,2987.0,NaN,NaN,996.0


In [ ]:
# FASE 2: TRANSFORM - Paso T2.5 (Corrección de Fechas)
# ==========================================
print("--- FASE 2: TRANSFORM (Conversión de Fechas) ---")

columnas_fechas = ['host_since', 'first_review', 'last_review']

for col in columnas_fechas:
    if col in df_listings.columns:
        # Convertimos a formato datetime. Si hay errores, se convertirán a NaT (Not a Time)
        df_listings[col] = pd.to_datetime(df_listings[col], errors='coerce')
        print(f"Columna '{col}' convertida exitosamente a datetime.")

# Chequeamos los tipos de datos para confirmar
display(df_listings[columnas_fechas].dtypes)

--- FASE 2: TRANSFORM (Conversión de Fechas) ---
Columna 'host_since' convertida exitosamente a datetime.
Columna 'first_review' convertida exitosamente a datetime.
Columna 'last_review' convertida exitosamente a datetime.


host_since      datetime64[ns]
first_review    datetime64[ns]
last_review     datetime64[ns]
dtype: object

In [7]:

# FASE 2: TRANSFORM - Paso T3 (Outliers y Nulos Lógicos)
# ==========================================
print("--- FASE 2: TRANSFORM (Paso T3) ---")

# 1. Imputación de nulos con lógica de negocio
# Si el anfitrión no puso monto de depósito o limpieza, asumimos que no cobra ($0)
cols_rellenar_cero = ['security_deposit', 'cleaning_fee']

for col in cols_rellenar_cero:
    if col in df_listings.columns:
        nulos_antes = df_listings[col].isnull().sum()
        df_listings[col] = df_listings[col].fillna(0)
        print(f"Columna '{col}': Se rellenaron {nulos_antes} valores nulos con $0.")

# 2. Tratamiento de Outliers en el Precio (Método IQR - Rango Intercuartílico)
Q1 = df_listings['price'].quantile(0.25)
Q3 = df_listings['price'].quantile(0.75)
IQR = Q3 - Q1

# Calculamos los bigotes del boxplot matemáticamente
limite_superior = Q3 + 1.5 * IQR
limite_inferior = max(10, Q1 - 1.5 * IQR) # Le ponemos un piso lógico de $10 la noche

print(f"\nLímites estadísticos para el precio: Min = ${limite_inferior:.2f}, Max = ${limite_superior:.2f}")

# Filtramos los outliers para quedarnos solo con precios normales
filas_antes_outliers = df_listings.shape[0]
df_listings = df_listings[(df_listings['price'] >= limite_inferior) & (df_listings['price'] <= limite_superior)]
filas_despues_outliers = df_listings.shape[0]

print(f"Se eliminaron {filas_antes_outliers - filas_despues_outliers} propiedades por tener precios atípicos (outliers).")

# 3. Borrado de nulos residuales en columnas estructurales
# Si falta la cantidad de baños, camas o dormitorios, la fila no nos sirve para medir rentabilidad
columnas_criticas = ['bedrooms', 'bathrooms', 'beds', 'accommodates']
df_listings = df_listings.dropna(subset=columnas_criticas)

print(f"\nDimensiones del dataset después de limpiar outliers y nulos críticos: {df_listings.shape}")

--- FASE 2: TRANSFORM (Paso T3) ---
Columna 'security_deposit': Se rellenaron 8195 valores nulos con $0.
Columna 'cleaning_fee': Se rellenaron 6785 valores nulos con $0.

Límites estadísticos para el precio: Min = $10.00, Max = $6206.50
Se eliminaron 2083 propiedades por tener precios atípicos (outliers).

Dimensiones del dataset después de limpiar outliers y nulos críticos: (21448, 94)


In [8]:

# FASE 2: TRANSFORM - Paso T4 (Normalización y Escalado)
# ==========================================
print("--- FASE 2: TRANSFORM (Paso T4) ---")

# Elegimos la variable a escalar
col_a_escalar = 'number_of_reviews'

# Verificamos que exista para no romper el código
if col_a_escalar in df_listings.columns:
    # Aplicamos la fórmula matemática Min-Max de forma manual y eficiente con Pandas
    min_val = df_listings[col_a_escalar].min()
    max_val = df_listings[col_a_escalar].max()
    
    # Creamos una nueva columna para no perder el dato original
    df_listings['number_of_reviews_scaled'] = (df_listings[col_a_escalar] - min_val) / (max_val - min_val)
    
    print(f"Columna '{col_a_escalar}' normalizada con éxito (Min-Max Scaler).")
    
    # Mostramos un antes y un después para comprobar
    display(df_listings[[col_a_escalar, 'number_of_reviews_scaled']].head())

--- FASE 2: TRANSFORM (Paso T4) ---
Columna 'number_of_reviews' normalizada con éxito (Min-Max Scaler).


,number_of_reviews,number_of_reviews_scaled
0,26,0.052
1,20,0.040
2,1,0.002
3,0,0.000
4,66,0.132


In [9]:

# FASE 2: TRANSFORM - Paso T5 (Integración de Tablas)
# ==========================================
print("--- FASE 2: TRANSFORM (Paso T5) ---")

# Vamos a usar df_calendar para calcular el porcentaje de disponibilidad real de cada propiedad
# 1. Convertimos la columna 'available' (que viene con 't'/'f' de true/false) a 1 y 0 numérico
if 'available' in df_calendar.columns:
    df_calendar['available_numeric'] = np.where(df_calendar['available'] == 't', 1, 0)

    # 2. Agrupamos por propiedad (listing_id) para sacar el promedio de días disponibles
    disponibilidad_anual = df_calendar.groupby('listing_id')['available_numeric'].mean().reset_index()
    
    # Renombramos para que sea claro
    disponibilidad_anual.rename(columns={'available_numeric': 'porcentaje_disponibilidad'}, inplace=True)

    # 3. Unimos (Merge / Left Join) este nuevo dato a nuestro dataset principal
    df_listings = pd.merge(df_listings, disponibilidad_anual, left_on='id', right_on='listing_id', how='left')

    # Limpiamos la columna extra que queda del cruce
    if 'listing_id' in df_listings.columns:
        df_listings = df_listings.drop(columns=['listing_id'])

    print("¡Integración exitosa! Se cruzaron los datos de calendar.csv con listings.csv")
else:
    print("No se encontró la columna 'available' en calendar.")

print(f"Dimensiones finales después de integrar: {df_listings.shape}")

--- FASE 2: TRANSFORM (Paso T5) ---
¡Integración exitosa! Se cruzaron los datos de calendar.csv con listings.csv
Dimensiones finales después de integrar: (21448, 96)


In [12]:
df_listings.sample(15)

,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,number_of_reviews_scaled,porcentaje_disponibilidad
3670,12755279,https://www.airbnb.com/rooms/12755279,20200426042522,2020-04-27,Hermoso depto de categoria,"seguridad, pileta, sauna, jacuzzy, parrilla, g...",NaN,"seguridad, pileta, sauna, jacuzzy, parrilla, g...",none,NaN,...,strict_14_with_grace_period,f,f,99,99,0,0,NaN,0.000,1.000000
7729,24395758,https://www.airbnb.com/rooms/24395758,20200426042522,2020-04-26,Departamento Buenos Aires 2 ambientes,"Cómodo y luminoso departamento de 2 ambientes,...","Cómodo y luminoso departamento de 2 ambientes,...","Cómodo y luminoso departamento de 2 ambientes,...",none,NaN,...,strict_14_with_grace_period,f,f,1,1,0,0,NaN,0.000,0.980822
4850,16715540,https://www.airbnb.com/rooms/16715540,20200426042522,2020-04-26,Amplio y cómodo Depto bien ubicado. Inolvidable.,El departamento se destaca por su amplio livin...,El departamento se puede utilizar en su totali...,El departamento se destaca por su amplio livin...,none,Ubicado en el corazón de Belgrano y Nuñez. A ...,...,flexible,f,f,1,1,0,0,2.75,0.040,0.000000
14842,35222637,https://www.airbnb.com/rooms/35222637,20200426042522,2020-04-26,"Departamento en Congreso, Montserrat.","Departamento muy luminoso, cómodo, bien ubicad...",El departamento cuenta con espacios muy amplio...,"Departamento muy luminoso, cómodo, bien ubicad...",none,"Se encuentra en el barrio de Montserrat, en la...",...,strict_14_with_grace_period,f,f,1,1,0,0,4.27,0.092,0.939726
11129,30694620,https://www.airbnb.com/rooms/30694620,20200426042522,2020-04-26,Beautiful apartment in RECOLETA Security guard...,Edificio Premier Arenales. Departamento de 1 ...,NaN,Edificio Premier Arenales. Departamento de 1 ...,none,La mejor zona de Buenos Aires!,...,strict_14_with_grace_period,f,f,1,1,0,0,0.85,0.028,0.490411
11422,31072870,https://www.airbnb.com/rooms/31072870,20200426042522,2020-04-26,Hermoso depto en el corazón de Palermo chico,Hermoso y cálido departamento totalmente recic...,NaN,Hermoso y cálido departamento totalmente recic...,none,NaN,...,strict_14_with_grace_period,f,f,1,1,0,0,NaN,0.000,0.487671
12014,31794850,https://www.airbnb.com/rooms/31794850,20200426042522,2020-04-26,ALGO MUY ESPECIAL EN LO MEJOR DE BUENOS AIRES,Mi apto. cómodo y céntrico está totalmente equ...,"Very quiet, luminous apartment, with a wide op...",Mi apto. cómodo y céntrico está totalmente equ...,none,Multiplex around the corner. Many restaurants...,...,flexible,f,f,2,2,0,0,0.07,0.002,0.000000
18203,39881087,https://www.airbnb.com/rooms/39881087,20200426042522,2020-04-26,Excelente mono ambiente sobre Av. Santa Fe,Amplio mono ambiente divisible. Excelente ubic...,NaN,Amplio mono ambiente divisible. Excelente ubic...,none,A metros de la Av. 9 de Julio y Plaza San Martin,...,flexible,f,f,2,2,0,0,NaN,0.000,0.243836
11679,31381208,https://www.airbnb.com/rooms/31381208,20200426042522,2020-04-26,Studio in Palermo. Comfortably experience BA!,"Nuevo, luminoso y cálido estudio, muy bien equ...","Funcional, nuevo y luminoso. Totalmente equipa...","Nuevo, luminoso y cálido estudio, muy bien equ...",none,Barrio tranquilo y a la vez con una buena ofer...,...,flexible,f,f,1,1,0,0,0.59,0.018,0.000000
9066,27902337,https://www.airbnb.com/rooms/27902337,20200426042522,2020-04-27,Cozy and luminous apartment in Palermo Chico,My beautiful and comfortable apartment with ex...,NaN,My beautiful and comfortable apartment with ex...,none,NaN,...,strict_14_with_grace_period,f,f,1,1,0,0,0.51,0.020,0.989041


In [14]:
# FASE 2: TRANSFORM - Paso T6 (Limpieza de columnas y nulos)
# ==========================================
print("--- FASE 2: TRANSFORM (Paso T6) ---")

# 1. Trabajamos sobre una copia de tu dataframe actual para mayor seguridad
df_listings_clean = df_listings.copy()

# 2. SELECCIÓN DE CARACTERÍSTICAS (Poda de columnas no analíticas)
columnas_basura = [
    'listing_url', 'scrape_id', 'last_scraped', 'source', 'name', 
    'description', 'neighborhood_overview', 'picture_url', 'host_url', 
    'host_name', 'host_about', 'host_thumbnail_url', 'host_picture_url', 
    'calendar_last_scraped', 'calendar_updated', 'license'
]

# Filtramos para borrar solo las que existen en df_listings_clean
columnas_a_eliminar = [col for col in columnas_basura if col in df_listings_clean.columns]
df_listings_clean.drop(columns=columnas_a_eliminar, inplace=True)
print(f"✔️ Se eliminaron {len(columnas_a_eliminar)} columnas no analíticas.")

# 3. PURGA DE NULOS MASIVOS (Eliminar columnas con más del 60% de vacíos)
porcentaje_nulos = (df_listings_clean.isnull().sum() / len(df_listings_clean)) * 100
columnas_muy_vacias = porcentaje_nulos[porcentaje_nulos > 60].index.tolist()


if columnas_muy_vacias:
    df_listings_clean.drop(columns=columnas_muy_vacias, inplace=True)
    print(f"✔️ Se eliminaron {len(columnas_muy_vacias)} columnas con >60% de nulos.")
else:
    print("✔️ Ninguna columna superó el 60% de nulos.")

# 4. REPORTE FINAL DE LA FASE
print(f"\nDimensiones tras la limpieza: {df_listings_clean.shape}")
print("\nTop 10 columnas con nulos restantes (para definir imputación):")
nulos_restantes = df_listings_clean.isnull().sum()
print(nulos_restantes[nulos_restantes > 0].sort_values(ascending=False).head(10))


--- FASE 2: TRANSFORM (Paso T6) ---
✔️ Se eliminaron 14 columnas no analíticas.
✔️ Ninguna columna superó el 60% de nulos.

Dimensiones tras la limpieza: (21448, 82)

Top 10 columnas con nulos restantes (para definir imputación):
interaction                    8534
transit                        7958
space                          6321
host_response_time             6120
host_response_rate             6120
review_scores_communication    5855
review_scores_value            5854
review_scores_accuracy         5854
review_scores_cleanliness      5854
review_scores_checkin          5853
dtype: int64


In [16]:
df_listings.head()


,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,number_of_reviews_scaled,porcentaje_disponibilidad
0,11508,https://www.airbnb.com/rooms/11508,20200426042522,2020-04-26,Amazing Luxurious Apt-Palermo Soho,NaN,LUXURIOUS NEW APT: 1 BDRM- POOL/ GYM/ SPA/ 24-...,LUXURIOUS NEW APT: 1 BDRM- POOL/ GYM/ SPA/ 24-...,none,AREA: PALERMO SOHO Minutes walking distance fr...,...,strict_14_with_grace_period,f,f,1,1,0,0,0.27,0.052,1.0
1,12463,https://www.airbnb.com/rooms/12463,20200426042522,2020-04-26,Room in Recoleta - awesome location,My apartment is centrally located in Recoleta ...,This is a very comfortable pull-out sofa in th...,My apartment is centrally located in Recoleta ...,none,It's near the school of medicine so the street...,...,moderate,f,f,1,0,1,0,0.16,0.040,1.0
2,13095,https://www.airbnb.com/rooms/13095,20200426042522,2020-04-26,Standard Room at Palermo Viejo B&B,Palermo Viejo B&B is a typical home in one of ...,Standard room : Palermo Viejo Bed & Breakfast ...,Palermo Viejo B&B is a typical home in one of ...,none,NaN,...,strict_14_with_grace_period,f,f,7,0,7,0,0.06,0.002,1.0
3,13096,https://www.airbnb.com/rooms/13096,20200426042522,2020-04-26,Standard Room in Palermo Viejo B&B,Palermo Viejo B&B is a typical home in one of ...,Palermo Viejo Bed & Breakfast is located in a ...,Palermo Viejo B&B is a typical home in one of ...,none,NaN,...,strict_14_with_grace_period,f,f,7,0,7,0,NaN,0.000,1.0
4,13097,https://www.airbnb.com/rooms/13097,20200426042522,2020-04-26,Standard Room at Palermo Viejo B&B,Palermo Viejo B&B is a typical home in one of ...,Palermo Viejo Bed & Breakfast is located in a ...,Palermo Viejo B&B is a typical home in one of ...,none,NaN,...,strict_14_with_grace_period,f,f,7,0,7,0,1.89,0.132,1.0


In [17]:
df_listings_clean.head()

,id,summary,space,experiences_offered,transit,interaction,host_id,host_since,host_location,host_response_time,...,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,number_of_reviews_scaled,porcentaje_disponibilidad
0,11508,NaN,LUXURIOUS NEW APT: 1 BDRM- POOL/ GYM/ SPA/ 24-...,none,All major bus lines are nearby. Stop: Plaza It...,NaN,42762,2009-10-01,"New York, New York, United States",within a few hours,...,strict_14_with_grace_period,f,f,1,1,0,0,0.27,0.052,1.0
1,12463,My apartment is centrally located in Recoleta ...,This is a very comfortable pull-out sofa in th...,none,Just a block from the Facultad de Medicina sub...,I work out of my apartment so I am around duri...,48799,2009-10-28,"Danbury, Connecticut, United States",within a few hours,...,moderate,f,f,1,0,1,0,0.16,0.040,1.0
2,13095,Palermo Viejo B&B is a typical home in one of ...,Standard room : Palermo Viejo Bed & Breakfast ...,none,NaN,NaN,50994,2009-11-03,"Buenos Aires, Capital Federal, Argentina",within an hour,...,strict_14_with_grace_period,f,f,7,0,7,0,0.06,0.002,1.0
3,13096,Palermo Viejo B&B is a typical home in one of ...,Palermo Viejo Bed & Breakfast is located in a ...,none,NaN,NaN,50994,2009-11-03,"Buenos Aires, Capital Federal, Argentina",within an hour,...,strict_14_with_grace_period,f,f,7,0,7,0,NaN,0.000,1.0
4,13097,Palermo Viejo B&B is a typical home in one of ...,Palermo Viejo Bed & Breakfast is located in a ...,none,NaN,NaN,50994,2009-11-03,"Buenos Aires, Capital Federal, Argentina",within an hour,...,strict_14_with_grace_period,f,f,7,0,7,0,1.89,0.132,1.0


In [18]:
# FASE 2: TRANSFORM - Paso T7 (Imputación Estratégica de Nulos)
# ==========================================
print("--- FASE 2: TRANSFORM (Paso T7) ---")

# 1. Eliminar textos libres restantes (no aportan a modelos predictivos ni numéricos)
text_cols = ['interaction', 'transit', 'space']
columnas_texto_a_borrar = [col for col in text_cols if col in df_listings_clean.columns]
df_listings_clean = df_listings_clean.drop(columns=columnas_texto_a_borrar)
print(f"✔️ Se eliminaron columnas de texto libre: {columnas_texto_a_borrar}")

# 2. Imputar Métricas del Anfitrión
if 'host_response_time' in df_listings_clean.columns:
    df_listings_clean['host_response_time'] = df_listings_clean['host_response_time'].fillna('Desconocido')

if 'host_response_rate' in df_listings_clean.columns:
    # Limpiamos el símbolo '%' y convertimos a número (float)
    df_listings_clean['host_response_rate'] = df_listings_clean['host_response_rate'].astype(str).str.replace('%', '', regex=False)
    df_listings_clean['host_response_rate'] = pd.to_numeric(df_listings_clean['host_response_rate'], errors='coerce')
    # Imputamos con la mediana
    mediana_response = df_listings_clean['host_response_rate'].median()
    df_listings_clean['host_response_rate'] = df_listings_clean['host_response_rate'].fillna(mediana_response)
    print("✔️ Se limpió 'host_response_rate' y se imputaron nulos con la mediana.")

# 3. Imputar Puntajes de Reseñas (con la mediana para no sesgar hacia abajo)
review_cols = [col for col in df_listings_clean.columns if col.startswith('review_scores_')]
for col in review_cols:
    mediana_score = df_listings_clean[col].median()
    df_listings_clean[col] = df_listings_clean[col].fillna(mediana_score)

print(f"✔️ Se imputaron nulos en {len(review_cols)} columnas de reseñas usando la mediana.")

# 4. REPORTE FINAL DE LA FASE TRANSFORM
print(f"\n--- Dimensiones listas de df_listings_clean: {df_listings_clean.shape} ---")
nulos_finales = df_listings_clean.isnull().sum()
nulos_sobrantes = nulos_finales[nulos_finales > 0]

if len(nulos_sobrantes) > 0:
    print("\nQuedan algunos nulos menores en:")
    print(nulos_sobrantes.sort_values(ascending=False).head(5))
else:
    print("\n¡Dataset 100% libre de nulos críticos!")

--- FASE 2: TRANSFORM (Paso T7) ---
✔️ Se eliminaron columnas de texto libre: ['interaction', 'transit', 'space']
✔️ Se limpió 'host_response_rate' y se imputaron nulos con la mediana.
✔️ Se imputaron nulos en 7 columnas de reseñas usando la mediana.

--- Dimensiones listas de df_listings_clean: (21448, 79) ---

Quedan algunos nulos menores en:
first_review            5491
reviews_per_month       5491
last_review             5491
zipcode                 4549
host_acceptance_rate    4505
dtype: int64


In [19]:
# FASE 2: TRANSFORM - Paso T8 (Cierre de Nulos)
# ==========================================
print("--- FASE 2: TRANSFORM (Paso T8) ---")

# 1. Propiedades sin reseñas (5491 nulos)
if 'reviews_per_month' in df_listings_clean.columns:
    df_listings_clean['reviews_per_month'] = df_listings_clean['reviews_per_month'].fillna(0)
    print("✔️ 'reviews_per_month' rellenado con 0 para propiedades sin reseñas.")

# 2. Tasa de aceptación del anfitrión
if 'host_acceptance_rate' in df_listings_clean.columns:
    df_listings_clean['host_acceptance_rate'] = df_listings_clean['host_acceptance_rate'].astype(str).str.replace('%', '', regex=False)
    df_listings_clean['host_acceptance_rate'] = pd.to_numeric(df_listings_clean['host_acceptance_rate'], errors='coerce')
    mediana_acceptance = df_listings_clean['host_acceptance_rate'].median()
    df_listings_clean['host_acceptance_rate'] = df_listings_clean['host_acceptance_rate'].fillna(mediana_acceptance)
    print("✔️ 'host_acceptance_rate' limpiada e imputada con la mediana.")

# 3. Código Postal (Zipcode)
if 'zipcode' in df_listings_clean.columns:
    df_listings_clean['zipcode'] = df_listings_clean['zipcode'].fillna('Desconocido')
    print("✔️ 'zipcode' rellenado con 'Desconocido'.")

# REPORTE FINAL DE LIMPIEZA
nulos_finales = df_listings_clean.isnull().sum()
nulos_sobrantes = nulos_finales[nulos_finales > 0]

print("\n--- RESUMEN FINAL DE NULOS ---")
if len(nulos_sobrantes) > 0:
    print(nulos_sobrantes)
    print("\nNota Analítica: Es totalmente correcto que las fechas de reseñas tengan nulos. Representan propiedades sin historial.")
else:
    print("¡Dataset 100% limpio!")

# ==========================================
# FASE 3: LOAD (Exportación del Dataset Listo)
# ==========================================
print("\n--- FASE 3: LOAD ---")
# Guardamos el dataset final limpio. 
# Reemplazá '../Data/' por la ruta de tu carpeta si es distinta.
ruta_exportacion = '../Data/airbnb_limpio_final.csv'
df_listings_clean.to_csv(ruta_exportacion, index=False)
print(f"✔️ ¡Éxito! Dataset guardado y listo para analizar en: {ruta_exportacion}")

--- FASE 2: TRANSFORM (Paso T8) ---
✔️ 'reviews_per_month' rellenado con 0 para propiedades sin reseñas.
✔️ 'host_acceptance_rate' limpiada e imputada con la mediana.
✔️ 'zipcode' rellenado con 'Desconocido'.

--- RESUMEN FINAL DE NULOS ---
summary                      1112
host_since                      3
host_location                  98
host_is_superhost               3
host_neighbourhood           2579
host_listings_count             3
host_total_listings_count       3
host_verifications              3
host_has_profile_pic            3
host_identity_verified          3
city                          537
state                         144
market                         20
first_review                 5491
last_review                  5491
dtype: int64

Nota Analítica: Es totalmente correcto que las fechas de reseñas tengan nulos. Representan propiedades sin historial.

--- FASE 3: LOAD ---
✔️ ¡Éxito! Dataset guardado y listo para analizar en: ../Data/airbnb_limpio_final.csv
